# Airport Parking Efficiency Monte Carlo Simulation

This notebook compares two parking methods for an airport parking lot:

- **Directed assignment**: the system sends the driver to a suitable open stall.
- **Sequential search**: the driver searches through stalls in sequence until a suitable stall is found.

The lot defaults to **50 x 100 spaces**, but the parameters below are easy to change.

In [ ]:
from pathlib import Path
from IPython.display import SVG, display
import parking_efficiency_simulation as sim

## Parameters

Edit this cell to change lot size, occupancy range, simulation count, or output folder.

In [ ]:
rows = 50
cols = 100
start_occupancy = 0.50
end_occupancy = 0.95
occupancy_step = 0.05
trials_per_condition = 100
seed = 20260331
output_dir = Path("outputs_notebook_50_to_95_100trials")

output_dir.mkdir(parents=True, exist_ok=True)
occupancy_levels = list(sim.frange(start_occupancy, end_occupancy, occupancy_step))
occupancy_levels

## Run Simulation

In [ ]:
all_results = []

for index, occupancy_ratio in enumerate(occupancy_levels):
    condition_results = sim.run_condition(
        rows=rows,
        cols=cols,
        occupancy_ratio=occupancy_ratio,
        trials=trials_per_condition,
        seed=seed + index,
    )
    all_results.extend(condition_results)

summary_rows = sim.summarize_results(all_results)

trials_path = output_dir / "parking_efficiency_trials.csv"
summary_path = output_dir / "parking_efficiency_summary.csv"
plot_path = output_dir / "parking_efficiency_plot.svg"
ci_plot_path = output_dir / "parking_efficiency_improvement_ci_plot.svg"

sim.write_trial_results(trials_path, all_results)
sim.write_summary(summary_path, summary_rows)
sim.render_svg_plot(plot_path, summary_rows)
sim.render_improvement_ci_plot(ci_plot_path, summary_rows)

print(f"Trials written to: {trials_path}")
print(f"Summary written to: {summary_path}")
print(f"Plot written to: {plot_path}")
print(f"CI plot written to: {ci_plot_path}")

## Summary Table

In [ ]:
summary_rows

## Plot

In [ ]:
display(SVG(filename=str(plot_path)))

## Quick Text Summary

In [ ]:
for row in summary_rows:
    print(
        f"{row['occupancy_pct']:.0f}% occupancy | "
        f"directed distance={row['avg_directed_distance']:.2f} | "
        f"sequential distance={row['avg_sequential_distance']:.2f} | "
        f"improvement={row['avg_improvement_pct']:.2f}%"
    )

## Notes

- `rows` and `cols` control parking-lot size.
- `trials_per_condition` can be reduced for speed or increased for smoother Monte Carlo averages.
- Vehicle-size and stall-size mixes are randomly generated inside each Monte Carlo trial.
- Directed assignment now uses Dijkstra's algorithm on a parking-lot graph with a central entry and center pathway.
- The parking field is split into equal left and right sections.
- The current sequential-search baseline searches the left section first and only switches to the right section if no suitable stall is found on the left.